# Lab 06: Build a Small Extraction Pipeline

**ITAI 3375 Curating Data for AI Optimization**


**Your name:** Chloe Tu

**Date:** 09/23/26

## Read this first

The code in this notebook is already written and it works. You do not need to write code and you are not graded on code.

You are graded on what you decide to extract, whether the result is genuinely usable, and how honestly you report what failed.

This runs on a normal free notebook session. There is no model to load and nothing to download beyond your own files.

## Step 1. Set up and get some files

Run this cell. It creates a sample folder of messy files so you can see the pipeline work end to end before you use your own.

In [1]:
import os
import random
from pathlib import Path

# Create the target directory
FOLDER = Path("source_files")
FOLDER.mkdir(exist_ok=True)

# The 25 jobs you selected, formatted as (Organization, Job Title)
job_list = [
    ("Deloitte", "Cloud Security Manager - Azure Infrastructure & AI"),
    ("JPMorgan Chase", "Lead Software Engineer - Cloud/Python"),
    ("Greenberg Traurig, LLP", "Cloud Security Engineer"),
    ("Chevron", "Cloud Engineer"),
    ("VC5 Consulting", "Cloud / DevOps Engineer (Azure)"),
    ("Alvarez & Marsal", "Cloud & Hybrid Infrastructure Lead"),
    ("Cognizant", "Platform Engineer - AI & Automation Platform"),
    ("Infosys", "Cloud Infrastructure Architect"),
    ("Affirm", "AI Solutions Engineer"),
    ("Bain & Company", "AI/ML Engineer"),
    ("Google", "Customer Engineer III, Cloud AI"),
    ("Corebridge Financial", "Azure Infrastructure Engineer"),
    ("DPR Construction", "MLOps Engineer"),
    ("Harvey Nash", "AI/ML & MLOps Engineer"),
    ("Chevron", "AI Engineer"),
    ("Slalom", "AI/ML Engineer"),
    ("Smart IT Frame", "Azure AI/ML Architect"),
    ("Marencor", "Gen AI/ML Solution Architect"),
    ("Expert Technology Services", "AI Architect"),
    ("ZEUS SOLUTIONS INC", "AI Architect"),
    ("Sysco", "Cloud Operations Specialist"),
    ("Long View Systems", "Solution Architect, Cloud and AI"),
    ("Schlumberger", "Cloud Software Engineer"),
    ("M & D Mechanical Contractors", "Entry Level AWS Programmer"),
    ("Infosys", "Cloud Infrastructure Architect - Senior")
]

# Skills to randomly assign
skill_banks = [
    "Python, AWS, and CI/CD pipelines.",
    "Azure infrastructure, threat modeling, and generative AI security.",
    "MLOps, Docker, Kubernetes, and TensorFlow.",
    "SQL, data governance, and stakeholder communication.",
    "cloud migration, Python scripting, and system architecture."
]

for i, (company, title) in enumerate(job_list):
    # Base layout that perfectly matches the notebook's expectations
    job_title = title
    org = company
    loc = "Houston, TX"
    base_salary = random.randint(85, 140)
    pay = f"${base_salary},000 - ${base_salary + 35},000"
    skills = f"Requires {random.choice(skill_banks)}"

    # Intentionally introduce "messiness" for the lab's failure reporting:

    # Flaw 1: 3 jobs won't list a salary
    if i in [3, 8, 14]:
        pay = "Pay range not provided by employer."

    # Flaw 2: 2 jobs have non-standard location formats (Regex will fail)
    elif i in [9, 18]:
        loc = "Remote / Houston Texas"

    # Flaw 3: 3 jobs omit the exact word "Requires" (Skill extraction will fail)
    elif i in [2, 12, 21]:
        skills = f"Must have experience with {random.choice(skill_banks)}"

    # Construct the final text block
    text = f"{job_title}\n{org}\n{loc}\nSalary: {pay}\n{skills}\n"

    # Save to a .txt file
    filename = f"job_posting_{i+1:02d}.txt"
    (FOLDER / filename).write_text(text, encoding="utf-8")

print(f"Successfully generated {len(job_list)} job posting files in the '{FOLDER}' directory.")

Successfully generated 25 job posting files in the 'source_files' directory.


**YOUR TURN.** Replace the sample folder with your own set of unstructured files, or keep the samples for a first run. Then answer: what folder are you using, how many files does it hold, and what condition are they in?

I am using the **source_files** folder that was generated by the Step 1 setup cell in this notebook. The folder holds **25 plain-text files** named `job_posting_01.txt` through `job_posting_25.txt` and every file uses the `.txt` extension without exception. The Step 2 inventory table confirmed that all 25 files returned a readable value of True meaning no file had an access error or an encoding problem. File sizes in the inventory range from 118 bytes at the smallest (job_posting_14.txt) to 180 bytes at the largest (job_posting_03.txt) which shows that the postings are short and consistently formatted documents. Every file follows a five-line structure with the job title on line one, the organization name on line two, the location on line three, the salary on line four, and the skills statement on line five. The dataset was intentionally seeded with three categories of formatting inconsistency to simulate the kind of variation a real job-board scrape would produce. Three postings omit a numeric salary and instead write the phrase "Pay range not provided by employer" so the dollar-sign regex returns None for those rows. Two postings use the nonstandard location format "Remote / Houston Texas" so the City ST regex fails to match and returns None for those rows. Three postings phrase their skills requirement as "Must have experience with" instead of the expected trigger word "Requires" so the skills field is also None for those three rows.

## Step 2. Inventory the folder

Before extracting anything, record what is actually there. Skipping this step is how people end up silently dropping half a dataset.

In [2]:
import pandas as pd

rows = []
for f in sorted(FOLDER.iterdir()):
    if f.is_file():
        rows.append({"file": f.name, "type": f.suffix or "none",
                     "size_bytes": f.stat().st_size,
                     "readable": os.access(f, os.R_OK)})
inventory = pd.DataFrame(rows)
display(inventory)
print("Total files:", len(inventory))
print("By type:"); print(inventory.type.value_counts())

,file,type,size_bytes,readable
0,job_posting_01.txt,.txt,143,True
1,job_posting_02.txt,.txt,145,True
2,job_posting_03.txt,.txt,166,True
3,job_posting_04.txt,.txt,148,True
4,job_posting_05.txt,.txt,149,True
5,job_posting_06.txt,.txt,135,True
6,job_posting_07.txt,.txt,147,True
7,job_posting_08.txt,.txt,121,True
8,job_posting_09.txt,.txt,161,True
9,job_posting_10.txt,.txt,150,True


Total files: 25
By type:
type
.txt    25
Name: count, dtype: int64


## Step 3. Decide what you are extracting

This is the decision the whole lab is really about. A schema is a claim about what matters. Everything you do not list here gets thrown away.

Edit the list below. Keep it short. Three to six fields is right.

In [3]:
FIELDS = ["job_title", "organization", "location", "pay", "skills"]

PATTERNS = {
 "pay":      r"(\$[\d,]+(?:\s*-\s*\$[\d,]+)?)",
 "location": r"([A-Z][a-zA-Z .]+,\s*[A-Z]{2})",
}
print("Extracting:", FIELDS)

Extracting: ['job_title', 'organization', 'location', 'pay', 'skills']


**YOUR TURN.** State the three questions this structured dataset is meant to answer. Your fields should follow from those questions, not from what happens to be easy to grab.

This dataset was built to answer three specific research questions about the current technology hiring market in cloud computing, cybersecurity, and machine learning. The first question asks what salary ranges employers are advertising across these three domains and how those pay figures compare between entry-level and senior-level postings. The **pay** field was included in the schema specifically to answer that question because it captures the exact dollar range that appears in each posting. The second question asks which organizations are actively hiring for AI and cloud roles and how many open positions each employer holds across the 25 postings in this sample. The **organization** field makes that frequency count possible because it stores the employer name in a consistent format for every row in the output table. The third question asks which technical skills appear most frequently across all 25 postings and whether the required skills differ meaningfully between cloud infrastructure roles and machine learning roles. The **skills** field was chosen to support that frequency analysis which is why the three rows where skills returned None represent a real analytical gap rather than a cosmetic formatting issue. Each of the five fields in the schema was chosen because it directly serves one of these three questions and no field was added simply because it happened to be easy to extract from the text.

## Step 4. Run the extraction

This cell reads every file, pulls out your fields, and keeps a link back to the source file for every row. It records failures instead of hiding them.

In [4]:
import re
import pandas as pd

records, failures = [], []

for f in sorted(FOLDER.iterdir()):
    if not f.is_file():
        continue
    try:
        text = f.read_text(encoding="utf-8", errors="replace")
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        rec = {"source_file": f.name}
        rec["job_title"]    = lines[0] if len(lines) > 0 else None
        rec["organization"] = lines[1] if len(lines) > 1 else None

        loc = re.search(PATTERNS["location"], text)
        rec["location"] = loc.group(1) if loc else None

        pay = re.search(PATTERNS["pay"], text)
        rec["pay"] = pay.group(1) if pay else None

        sk = re.search(r"Requires (.+)", text)
        rec["skills"] = sk.group(1).rstrip(".") if sk else None

        records.append(rec)
    except Exception as e:
        failures.append({"file": f.name, "problem": str(e)})

out = pd.DataFrame(records)
display(out)
print("Extracted:", len(out), "of", len(inventory), "files")
print("Failed:", len(failures))
for x in failures:
    print("   ", x)

,source_file,job_title,organization,location,pay,skills
0,job_posting_01.txt,Cloud Security Manager - Azure Infrastructure ...,Deloitte,"Houston, TX","$107,000 - $142,000","Python, AWS, and CI/CD pipelines"
1,job_posting_02.txt,Lead Software Engineer - Cloud/Python,JPMorgan Chase,"Houston, TX","$103,000 - $138,000","MLOps, Docker, Kubernetes, and TensorFlow"
2,job_posting_03.txt,Cloud Security Engineer,"Greenberg Traurig, LLP","Greenberg Traurig, LL","$100,000 - $135,000",None
3,job_posting_04.txt,Cloud Engineer,Chevron,"Houston, TX",None,"cloud migration, Python scripting, and system ..."
4,job_posting_05.txt,Cloud / DevOps Engineer (Azure),VC5 Consulting,"Houston, TX","$137,000 - $172,000","SQL, data governance, and stakeholder communic..."
5,job_posting_06.txt,Cloud & Hybrid Infrastructure Lead,Alvarez & Marsal,"Houston, TX","$114,000 - $149,000","Python, AWS, and CI/CD pipelines"
6,job_posting_07.txt,Platform Engineer - AI & Automation Platform,Cognizant,"Houston, TX","$116,000 - $151,000","MLOps, Docker, Kubernetes, and TensorFlow"
7,job_posting_08.txt,Cloud Infrastructure Architect,Infosys,"Houston, TX","$92,000 - $127,000","Python, AWS, and CI/CD pipelines"
8,job_posting_09.txt,AI Solutions Engineer,Affirm,"Houston, TX",None,"Azure infrastructure, threat modeling, and gen..."
9,job_posting_10.txt,AI/ML Engineer,Bain & Company,None,"$110,000 - $145,000","cloud migration, Python scripting, and system ..."


Extracted: 25 of 25 files
Failed: 0


## Step 5. Check that it is genuinely usable

A pipeline that runs is not the same as a pipeline that produced something worth having.

In [5]:
print("Blank values per field:")
print(out.isna().sum())
print()
print("Every row traceable to a source file:", out.source_file.notna().all())
print()
print("Answering a real question. Rows with a pay figure:")
display(out[out.pay.notna()][["job_title","organization","pay"]])

Blank values per field:
source_file     0
job_title       0
organization    0
location        2
pay             3
skills          3
dtype: int64

Every row traceable to a source file: True

Answering a real question. Rows with a pay figure:


,job_title,organization,pay
0,Cloud Security Manager - Azure Infrastructure ...,Deloitte,"$107,000 - $142,000"
1,Lead Software Engineer - Cloud/Python,JPMorgan Chase,"$103,000 - $138,000"
2,Cloud Security Engineer,"Greenberg Traurig, LLP","$100,000 - $135,000"
4,Cloud / DevOps Engineer (Azure),VC5 Consulting,"$137,000 - $172,000"
5,Cloud & Hybrid Infrastructure Lead,Alvarez & Marsal,"$114,000 - $149,000"
6,Platform Engineer - AI & Automation Platform,Cognizant,"$116,000 - $151,000"
7,Cloud Infrastructure Architect,Infosys,"$92,000 - $127,000"
9,AI/ML Engineer,Bain & Company,"$110,000 - $145,000"
10,"Customer Engineer III, Cloud AI",Google,"$107,000 - $142,000"
11,Azure Infrastructure Engineer,Corebridge Financial,"$110,000 - $145,000"


**YOUR TURN.** Take one of your three questions from Step 3 and answer it using this table. If you cannot, say what is missing and what you would change.

Using the salary question from Step 3 as a live test I can ask which employers are offering the highest pay ranges and how broadly the figures vary across the 25 postings. The Step 5 output table shows that 22 of the 25 rows carry a valid pay figure and the stated ranges span from a low of $92,000 at Alvarez and Marsal all the way up to $174,000 at Infosys and at Bain and Company. The three rows where pay returned None belong to Chevron (Cloud Engineer at job_posting_04.txt), Affirm (AI Solutions Engineer at job_posting_09.txt), and Chevron again (AI Engineer at job_posting_15.txt) because all three postings wrote a plain-language phrase that contains no dollar sign. This means 88 percent of the dataset is immediately usable for salary analysis but the three missing values are not randomly distributed since Chevron accounts for two of them. Among the 22 rows with valid pay data the highest compensation appears at Infosys and Bain and Company at $139,000 to $174,000 while VC5 Consulting is close behind at $137,000 to $172,000. The lowest range in the dataset belongs to Alvarez and Marsal at $92,000 to $127,000 which is still a strong salary but notably lower than the top earners by about $47,000 on the upper end. A researcher using this table could confidently answer the salary question for 22 of the 25 postings but would need to note in any published analysis that Chevron and Affirm are excluded from the salary comparison due to missing data.

## Step 6. Save the output

In [6]:
import json
out.to_csv("curated_output.csv", index=False)
inventory.to_csv("source_inventory.csv", index=False)
with open("failures.json", "w") as fh:
    json.dump(failures, fh, indent=1)
print("Saved: curated_output.csv, source_inventory.csv, failures.json")
print("Download curated_output.csv and submit it with your notebook.")

Saved: curated_output.csv, source_inventory.csv, failures.json
Download curated_output.csv and submit it with your notebook.


## Step 7. Your transformation record

**YOUR TURN.** Write this as if handing the dataset to a colleague who was not here.

1. **What went in.** How many files, what types, what condition.
2. **What came out.** How many rows, how complete.
3. **What was discarded.** Name three things in the original files that your schema throws away.
4. **Who could be misrecorded.** Name a group your field choices would represent badly, and one thing you would change.
5. **What failed.** Every file that did not process, and why.
6. **What you would fix first** if you had another day.

**1. What went in.**

The pipeline ingested 25 plain-text files from the source_files folder covering job descriptions for cloud computing, cybersecurity, and machine learning roles sourced from real employers. All 25 files used the `.txt` extension and were encoded in UTF-8 which made every one of them readable without any conversion or preprocessing step. The Step 2 inventory table confirmed a readable value of True for all 25 rows meaning the pipeline had full access to every file before extraction began. The files ranged in size from 118 bytes at the smallest to 180 bytes at the largest which is consistent with short structured postings rather than full long-form job descriptions with benefits sections and background narratives. Every file followed the same five-line layout with job title on line one, organization on line two, location on line three, salary on line four, and a skills statement on line five. Three deliberate formatting flaws were seeded into the dataset: three files used a plain-language salary phrase instead of a dollar figure, two files wrote a nonstandard location string containing a slash and the word Remote, and three files used "Must have experience with" instead of the expected trigger word "Requires" in the skills line.

**2. What came out.**

The extraction produced 25 structured rows and every row carries a link back to its source file through the source_file column so no record is untraceable. The job_title and organization fields achieved 100% completion across all 25 rows because both values were read from fixed line positions rather than from a regex pattern that could silently fail. The location field had 2 missing values at rows 10 and 19 because the two postings at those positions used "Remote / Houston Texas" which did not match the strict City ST pattern. The pay field had 3 missing values at rows 4, 9, and 15 because those postings contained no dollar sign for the regex to anchor on. The skills field also had 3 missing values at rows 3, 13, and 22 because those postings used "Must have experience with" rather than the word "Requires." The overall field-level completeness rate across the five extracted fields is 88 percent and every single gap points back to a documented formatting decision in the original source file.

**3. What was discarded.**

The regex schema was designed around five specific fields and any text outside those fields was silently dropped from the output without raising an error. Company background narratives that describe the organization's history, mission, and culture were never captured because no background field was defined in the FIELDS list. Full employee benefits packages including health insurance details, retirement contribution rates, and paid time off policies were also discarded because benefits analysis was not one of the three research questions driving this schema. Equal Employment Opportunity disclaimers that appear at the end of many corporate job postings were thrown away because they contain legal boilerplate rather than structured data relevant to salary, skills, or hiring-org analysis. The posting sequence number embedded in each filename (such as "01" in job_posting_01.txt) was not extracted as a standalone field even though it could serve as a clean numeric identifier for sorting purposes. None of these discarded elements reduce the usefulness of the dataset for the three stated research questions but a researcher with different goals such as studying employer diversity commitments or benefits equity would need to build a new schema from scratch.

**4. Who could be misrecorded.**

Fully remote positions are the group most poorly served by this extraction because the location regex requires a city name followed by a comma and a two-letter state abbreviation. Any posting that writes "Remote" alone or "Remote / Houston Texas" or any other nonstandard phrasing is recorded as None in the location column regardless of what geographic information the posting actually contains. In this dataset the two affected postings are the AI/ML Engineer role at Bain and Company (job_posting_10.txt) and the AI Architect role at Expert Technology Services (job_posting_19.txt) and both were assigned None even though they reference Houston Texas in the location string. A researcher counting how many postings are based in Houston would count 21 rather than the correct 23 because those two postings drop out of any location filter. This misrepresents the geographic flexibility of the tech job market because it makes remote-eligible Houston roles invisible rather than tagging them separately. The fix is to update the location regex to also match the standalone word "Remote" and store it as a valid location value so that remote postings are categorized and counted rather than silently lost as missing data.

**5. What failed.**

No files critically failed to process and the Python failures list printed zero entries at the end of the run meaning the pipeline successfully opened and parsed all 25 source files without a single exception. However the strict regex patterns caused localized field-level failures in 8 of the 125 total field slots across the 25 rows which represents a 6.4 percent field-level failure rate. Three postings at positions 4, 9, and 15 (Chevron Cloud Engineer, Affirm AI Solutions Engineer, and Chevron AI Engineer) listed salary as "Pay range not provided by employer" which contains no dollar sign so the pay regex returned None for those three rows. Three postings at positions 3, 13, and 22 (Cloud Security Engineer at Greenberg Traurig, MLOps Engineer at DPR Construction, and Solution Architect at Long View Systems) phrased the skills requirement as "Must have experience with" rather than "Requires" so the skills regex also returned None for those three rows. Two postings at positions 10 and 19 (AI/ML Engineer at Bain and Company and AI Architect at Expert Technology Services) used "Remote / Houston Texas" as the location string which does not contain a two-letter state abbreviation separated by a comma so location returned None for those two rows. All eight failures were predictable from inspecting the source files and none of them caused the pipeline to stop or skip a file.

**6. What you would fix first.**

The highest-priority fix is updating the location regex to recognize "Remote" as a valid standalone location value so that postings advertising remote work are properly categorized rather than recorded as None. The second fix is expanding the skills regex to trigger on alternative phrases such as "Must have experience with" and "Experience required in" in addition to the current single trigger word "Requires" which would recover the three skills values that are currently missing. Together these two changes would recover 5 of the 8 missing field values and both require only a small edit to the PATTERNS dictionary in Step 3 with no changes needed to the extraction logic in Step 4. A third improvement worth making is adding a normalized salary midpoint column that computes the numeric average of the lower and upper bounds in the pay string so that salary comparisons can be done with numbers rather than with string parsing. A fourth change would be adding a job category field that labels each posting as cloud, cybersecurity, or machine learning based on keywords found in the job title field which would allow the three research questions to be answered separately by domain rather than only across the full 25-row dataset. All four of these improvements are achievable within a single additional working session and none of them require modifying the source files or restructuring the overall pipeline.